In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpnsp_pavl".


In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <stdio.h>
#include <stdlib.h>
#define N 8
// CUDA constants
__constant__ float dev_A[N][N];
// declaracion de funciones
// GLOBAL: funcion llamada desde el host y ejecutada en el device (kernel)

__global__ void traspuesta(float* dev_B)
{ // kernel lanzado con un solo bloque y NxN hilos
    int columna = threadIdx.x;
    int fila = threadIdx.y;
    int pos = columna + N * fila;

    // cada hilo coloca un elemento de la matriz final
    dev_B[pos] = dev_A[columna][fila];
}

// MAIN: rutina principal ejecutada en el host
int main(int argc, char** argv) {
    // declaraciones
    float* hst_A, * hst_B;
    float* dev_B;
    // reserva en el host
    hst_A = (float*)malloc(N * N * sizeof(float));
    hst_B = (float*)malloc(N * N * sizeof(float));
    // reserva en el device
    cudaMalloc((void**)&dev_B, N * N * sizeof(float));
    // inicializacion
    for (int i = 0; i < N * N; i++)
    {
        hst_A[i] = (float)i;
    }
    // copia de datos
    cudaMemcpyToSymbol(dev_A, hst_A, N * N * sizeof(float));
    // dimensiones del kernel
    dim3 Nbloques(1);
    dim3 hilosB(N, N);
    // llamada al kernel bidimensional de NxN hilos
    traspuesta <<< Nbloques, hilosB >>> (dev_B);
    // recogida de datos
    cudaMemcpy(hst_B, dev_B, N * N * sizeof(float), cudaMemcpyDeviceToHost);

    // impresion de resultados  printf("Resultado:\n");  printf("ORIGINAL:\n");
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            printf("%2.0f ", hst_A[j + i * N]);
        }
        printf("\n");
    }

    printf("TRASPUESTA:\n");
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            printf("%2.0f ", hst_B[j + i * N]);
        }
        printf("\n");
    }
}

 0  1  2  3  4  5  6  7 
 8  9 10 11 12 13 14 15 
16 17 18 19 20 21 22 23 
24 25 26 27 28 29 30 31 
32 33 34 35 36 37 38 39 
40 41 42 43 44 45 46 47 
48 49 50 51 52 53 54 55 
56 57 58 59 60 61 62 63 
TRASPUESTA:
 0  8 16 24 32 40 48 56 
 1  9 17 25 33 41 49 57 
 2 10 18 26 34 42 50 58 
 3 11 19 27 35 43 51 59 
 4 12 20 28 36 44 52 60 
 5 13 21 29 37 45 53 61 
 6 14 22 30 38 46 54 62 
 7 15 23 31 39 47 55 63 

